# Maximum Likelihood  #

The Phat distribution can be fitted to a univariate dataset by Maximum Likelihood Estimation (MLE). This is available through `Phat.fit`, which maximises the likelihood with `scipy.optimize` and reports standard errors from the observed information.

There is one major issue with the tails that must be considered.

First, let's fit the Phat distribution to our familiar series of daily S&P 500 index returns (1950 to November 2026; see `docs/data`).

In [1]:
import seaborn as sns; sns.set(style = 'whitegrid')

In [2]:
import pandas as pd
import phat as ph

sp_ret = pd.read_csv('../data/sp500_returns.csv', index_col='date', parse_dates=True)['return']

res = ph.Phat.fit(sp_ret)

In [3]:
res.params_dict

{'mu': 0.000571647274575266,
 'sigma': 0.003271635219148546,
 'xi_left': 0.0,
 'xi_right': 0.0}

In [4]:
print(res.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 64028.5212)
             estimate      std err
mu           0.000572     0.000049
sigma        0.003272     0.000027
xi_left      0.000000   (at bound)
xi_right     0.000000   (at bound)


The full MLE puts **both tail shapes at zero**, the lower bound of the search. On this sample the likelihood prefers exponential tails to the power-law tails we estimated with the [POT](pot_ref.ipynb) and [Hill Double Bootstrap techniques](dblbs.ipynb). An estimate on a bound has no valid Wald standard error, so none is reported.

There is a structural reason. In Phat the shape $\xi$ of a tail also fixes where its junction sits and how much probability lies beyond it, and the junctions are less than one $\sigma$ from the centre. So $\xi$ has to describe the shoulders of the distribution, where most of the observations are, as well as the extreme tail, and the likelihood is dominated by the shoulders. The result is thinner fitted tails that understate the risk of extreme events.

Instead, we can estimate the tail shapes separately and pass them as fixed values to `fit`. This leaves two free parameters: the location $\mu$ and the scale $\sigma$.

In [5]:
xi_left, xi_right = ph.two_tailed_hill_double_bootstrap(sp_ret, seed=42, pbar_kwargs={'disable': True})
xi_left, xi_right

(0.3192120423228642, 0.3279048642691963)

In [6]:
res2 = ph.Phat.fit(sp_ret, xi_left, xi_right)
print(res2.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 63243.4148)
             estimate      std err
mu           0.000591     0.000046
sigma        0.002956     0.000030
xi_left      0.319212      (fixed)
xi_right     0.327905      (fixed)


In [7]:
res2.params_dict

{'mu': 0.0005910273352607609, 'sigma': 0.0029558310203372036}

In [8]:
res.dist.std(), res2.dist.std(), sp_ret.std()

(0.010541345649623996, 0.019139025073904452, np.float64(0.009929074858842975))

With the tail shapes held at the bootstrap estimates the location barely moves and the scale falls by about 10%. The log-likelihood is lower, as it must be for a constrained fit.

The two fits imply different standard deviations, shown above next to the sample value. The full MLE is close to the sample; the two-stage fit, with its much heavier tails, implies a standard deviation almost twice the sample's. That is the trade: the two-stage fit carries the tail indices of the extreme-value estimators at the cost of fit in the centre.

## Estimating the junction ##

Both fits above use the implied junction, and both are strained by it: the full MLE gives up the power-law tails to fit the shoulders, and the two-stage fit keeps the tails and pays for it in the centre.

`Phat.fit` can estimate the junction instead (see [the free junction](thephat.ipynb)). `junction='symmetric'` adds one parameter, a junction distance `s` shared by both sides, in units of `sigma` from `mu`. `junction='free'` adds two, `s_left` and `s_right`.

In [9]:
res_sym = ph.Phat.fit(sp_ret, junction='symmetric')
print(res_sym.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 64321.7246)
             estimate      std err
mu           0.000568     0.000055
sigma        0.005645     0.000101
xi_left      0.085381     0.013960
xi_right     0.051460     0.012873
s            0.785332     0.027894


In [10]:
res_free = ph.Phat.fit(sp_ret, junction='free')
print(res_free.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 64324.9480)
             estimate      std err
mu           0.000664     0.000067
sigma        0.005659     0.000102
xi_left      0.062812     0.015225
xi_right     0.078618     0.018397
s_left       0.746912     0.029808
s_right      0.838210     0.037779


The junction is estimated well inside its bounds, with a small standard error, and the tail shapes come off the zero bound.

The implied junction is the special case $\lambda = 1$ of the free one, so the fits are nested and can be compared by their log-likelihoods. Below: the number of estimated parameters, the log-likelihood, the gain over the implied junction, AIC and BIC.

In [11]:
fits = {'implied': res, 'symmetric': res_sym, 'free': res_free}
pd.DataFrame({
    'parameters': {k: len(v.params) for k, v in fits.items()},
    'log-likelihood': {k: v.llf for k, v in fits.items()},
    'gain over implied': {k: v.llf - res.llf for k, v in fits.items()},
    'AIC': {k: v.aic for k, v in fits.items()},
    'BIC': {k: v.bic for k, v in fits.items()},
}).round(1)

,parameters,log-likelihood,gain over implied,AIC,BIC
implied,4,64028.5,0.0,-128049.0,-128017.6
symmetric,5,64321.7,293.2,-128633.4,-128594.1
free,6,64324.9,296.4,-128637.9,-128590.7


One extra parameter is worth several hundred log-likelihood points on this sample; a chi-squared test of $\lambda = 1$ would need about two. A separate distance for each side adds little more, and BIC prefers the shared one.

Where the junction ends up, and how much of the distribution is Gaussian as a result:

In [12]:
pd.DataFrame({
    name: {
        's_left': d.s_left, 's_right': d.s_right,
        'lambda_left': d.lam_left, 'lambda_right': d.lam_right,
        'P(between the junctions)': d.body_mass,
        'xi_left': d.xi_left, 'xi_right': d.xi_right,
        'standard deviation': d.std(),
    } for name, d in {'implied': res.dist, 'symmetric': res_sym.dist, 'free': res_free.dist}.items()
}).round(4)

,implied,symmetric,free
s_left,0.3722,0.7853,0.7469
s_right,0.3722,0.7853,0.8382
lambda_left,1.0000,0.4051,0.4295
lambda_right,1.0000,0.3924,0.3613
P(between the junctions),0.1764,0.4801,0.4840
xi_left,0.0000,0.0854,0.0628
xi_right,0.0000,0.0515,0.0786
standard deviation,0.0105,0.0098,0.0098


The likelihood moves the junctions out to roughly twice the implied distance and gives the Pareto pieces less than half the weight that $\lambda = 1$ forces on them.

The fitted tail shapes are still far below the Hill estimates. That is a property of maximum likelihood on the whole sample, not of the junction: almost all observations are in the centre, so the centre decides. The two-stage estimator is available with a free junction too. The tail shapes are held at the bootstrap estimates and the location, the scale and the junctions are fitted:

In [13]:
res2_free = ph.Phat.fit(sp_ret, xi_left, xi_right, junction='free')
print(res2_free.summary())

Phat maximum-likelihood fit  (nobs = 19346, llf = 64231.5012)
             estimate      std err
mu           0.000609     0.000060
sigma        0.006426     0.000075
s_left       1.163732     0.024653
s_right      1.276547     0.027027
xi_left      0.319212      (fixed)
xi_right     0.327905      (fixed)


Finally, the quantiles each fit implies against the empirical quantiles of the sample, in percent. The two-stage fit with the implied junction puts the 0.1% quantile at more than twice the empirical one; with a free junction the gap is less than half as large, with the same bootstrap tail shapes. In-sample quantiles cannot say which tail is right about losses not yet seen: that takes an out-of-sample comparison.

In [14]:
levels = [0.001, 0.01, 0.05, 0.95, 0.99, 0.999]
table = pd.DataFrame({
    'empirical': sp_ret.quantile(levels).values,
    'MLE, implied': res.dist.ppf(levels),
    'MLE, symmetric': res_sym.dist.ppf(levels),
    'two-stage, implied': res2.dist.ppf(levels),
    'two-stage, free': res2_free.dist.ppf(levels),
}, index=levels)
(table * 100).round(2)

,empirical,"MLE, implied","MLE, symmetric","two-stage, implied","two-stage, free"
0.001,-5.95,-5.09,-5.42,-13.46,-8.45
0.010,-2.70,-3.07,-2.89,-5.15,-3.22
0.050,-1.48,-1.65,-1.43,-2.07,-1.39
0.950,1.47,1.77,1.48,2.21,1.41
0.990,2.58,3.18,2.79,5.36,2.95
0.999,5.13,5.20,4.92,14.03,7.59
